In [ ]:
import numpy as np
import torch

from iensf import NoiseSchedule, ScoreRep, ReverseSampler, Langevin, post_score, post_score_static
from torch.distributions.multivariate_normal import MultivariateNormal
from da_utils import KF_update, EnKF_update, KF_with_sample
import matplotlib.pyplot as plt
import torch.nn as nn
import functools

torch.set_default_dtype(torch.float64)
dtype = torch.float64
mse = nn.MSELoss()



In [ ]:
prior_mean_true = torch.tensor([0,0], dtype=dtype)
prior_cov_true = torch.tensor([[0.5, -0.4],[-0.4,0.5]], dtype=dtype)
dim_x = prior_cov_true.shape[0]
prior_dist = MultivariateNormal(loc=prior_mean_true, covariance_matrix=prior_cov_true)
Nx = 1000
prior_sample = prior_dist.rsample((Nx,))


def obs_model(x):
    """
    :param x:(N, dim_x)
    :param H: (dim_obs, dim_x)
    :return: (N, dim_obs)
    """
    x1 = x[:,[0]]
    # x2 = x[:,[1]]
    obs = x1
    return obs

obs_value =  torch.tensor([3.])
dim_obs = obs_value.shape[0]
obs_sigma = 0.1
obs_cov = torch.eye(dim_obs) * obs_sigma**2

# setup diffusion model
ns = NoiseSchedule(ns_type='linear',eps_a=0.00001, eps_b=0.00001)
dm = ReverseSampler(noise_schedule=ns)
score_rep = ScoreRep(dm=dm, dim_x=dim_x, obs_model=obs_model, obs_sigma=obs_sigma)
score_rep.set_obs_value(obs_value=obs_value)

Nt = 100

In [ ]:
x_min, x_max = 0,5
y_min, y_max = -5,0

N_mesh = 100
x_1d = torch.linspace(x_min, x_max, N_mesh)
y_1d = torch.linspace(y_min, y_max, N_mesh)
x_2d, y_2d = torch.meshgrid(x_1d, y_1d, indexing='ij')

xy_2d = torch.stack([x_2d.flatten() , y_2d.flatten()], dim=1)

log_l = score_rep.log_likelihood(xy_2d)
log_l -= torch.max(log_l)
l_2d = torch.exp(log_l)



log_pdf_prior = prior_dist.log_prob(xy_2d)
log_pdf_post = log_pdf_prior + log_l

pdf_prior_2d = torch.exp(log_pdf_prior - log_pdf_prior.max())
pdf_post_2d = torch.exp(log_pdf_post - log_pdf_post.max())

max_l, loc_idx = torch.max(pdf_post_2d,dim=0)
xy_max = xy_2d[loc_idx]

# plot 
cmap = 'viridis'
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(10,3))

ax = axs
data = pdf_prior_2d.reshape(N_mesh,N_mesh)
ax[0].pcolormesh(x_2d, y_2d, data, cmap=cmap,vmin=0, vmax=1)
ax[0].set_title('Prior')

data = l_2d.reshape(N_mesh,N_mesh)
ax[1].pcolormesh(x_2d, y_2d, data, cmap=cmap,vmin=0, vmax=1)
ax[1].set_title(f'Likelihood (r={obs_value.item():.1f})')

data = pdf_post_2d.reshape(N_mesh,N_mesh)
ax[2].pcolormesh(x_2d, y_2d, data, cmap=cmap,vmin=0, vmax=1)
ax[2].set_title('Posterior')
# plt.savefig('bias_correction_pdf.pdf', dpi=200)
# plt.savefig('bias_correction_pdf.png', dpi=200)
plt.show()

In [ ]:
# pf sample 
N_pf = 100000
prior_sample_pf = prior_dist.rsample((N_pf,))
log_l = score_rep.log_likelihood(prior_sample_pf)
log_l -= torch.max(log_l)
l_pf = torch.exp(log_l)
p = l_pf/torch.sum(l_pf)
N_boot = Nx
idx = np.random.choice(N_pf, N_boot, p=p.numpy(), replace=True)
post_sample_pf = prior_sample_pf[idx,:]

# check result 
data = pdf_post_2d.reshape(N_mesh,N_mesh)
plt.pcolormesh(x_2d, y_2d, data, cmap=cmap,vmin=0, vmax=1)
plt.plot(*post_sample_pf.T,'r.',alpha=0.1)
plt.plot()
plt.title('posterior(PF)')
plt.show()

In [ ]:
# EnKF 
post_sample_enkf = EnKF_update(prior_sample, obs_model, obs_value, obs_sigma)


# check result 
data = pdf_post_2d.reshape(N_mesh,N_mesh)
plt.pcolormesh(x_2d, y_2d, data, cmap=cmap,vmin=0, vmax=1)
plt.plot(*post_sample_enkf.T,'r.',alpha=0.01)
plt.plot()
plt.title('posterior(EnKF)')
plt.show()

enkf_mean = torch.mean(post_sample_enkf, dim=0)
enkf_cov = torch.cov(post_sample_enkf.T)

In [ ]:
x_T = torch.randn(Nx, dim_x)
x_T = (x_T - torch.mean(x_T, dim=0))/torch.std(x_T, dim=0)
solver_all = ['SDE_euler', 'ODE_euler', 'DPM_solver']
def post_score(zt,t, score_prior, score_likelihood):
    return score_prior(zt,t) + score_likelihood(zt,t)
score_prior_fn = functools.partial(score_rep.score_gaussian_diffusion,
                                   prior_mean=prior_mean_true, prior_cov=prior_cov_true)

Nt = 200

In [ ]:
# damp: 1-t
def damp_fn(t):
    return 1-t
score_l1 = functools.partial(score_rep.score_likelihood_1_generic, damp_fn=damp_fn)

##############################################################################
# sampling module
post_score_fn = functools.partial(post_score, score_prior=score_prior_fn, score_likelihood=score_l1)
post_sample_score_original = dm.sample_gen(x_T=x_T, score_fn=post_score_fn, Nt=Nt, solver_type=solver_all[0])


# check result
data = pdf_post_2d.reshape(N_mesh,N_mesh)
plt.pcolormesh(x_2d, y_2d, data, cmap=cmap,vmin=0, vmax=1)
plt.plot(*post_sample_score_original.T,'r.',alpha=0.1)
plt.title('posterior(original)')
plt.show()

In [ ]:
# iteration based sampling 
eta1 = 0.5
eta2 = 0.5

current_sample = prior_sample.clone()
mu_star = torch.mean(current_sample, dim=0)
cov_star = torch.cov(current_sample.T)

sample_iter = []
N_iter = 10
for i in range(N_iter):
    
    score_l4 = functools.partial(score_rep.score_likelihood_3_full_generic,
                                 prior_mean=prior_mean_true, prior_cov=prior_cov_true,
                                 post_mean=mu_star, post_cov=cov_star)

    post_score_fn = functools.partial(post_score, score_prior=score_prior_fn, score_likelihood=score_l4)
    current_sample = dm.sample_gen(x_T=x_T, score_fn=post_score_fn, Nt=Nt, solver_type=solver_all[0])
    
    sample_iter.append(current_sample)
    
    current_mean = torch.mean(current_sample, dim=0)
    current_cov = torch.cov(current_sample.T)
    
    print(f'iter {i}: mean diff {torch.norm(current_mean-enkf_mean):.4e}, cov diff {torch.norm(current_cov - enkf_cov):.4e}')
    
    mu_star = (1-eta1)*mu_star + eta1*current_mean
    cov_star = (1-eta2)*cov_star + eta2*current_cov


for i in range(N_iter):
    plt.figure()
    data = pdf_post_2d.reshape(N_mesh,N_mesh)
    plt.pcolormesh(x_2d, y_2d, data, cmap=cmap,vmin=0, vmax=1)
    plt.plot(*sample_iter[i].T,'r.',alpha=0.1)
    plt.title(f'posterior(iter {i})')
    plt.show()

In [ ]:
# KL and er 
def kl_divergence_gaussians(mu0, Sigma0, mu1, Sigma1):
    """
    Compute KL divergence D_KL(N(mu0,Sigma0) || N(mu1,Sigma1)).

    mu0, mu1: mean vectors (d,)
    Sigma0, Sigma1: covariance matrices (d,d)
    """
    d = mu0.shape[0]
    
    # Inverse and determinant
    inv_Sigma1 = np.linalg.inv(Sigma1)
    det_Sigma0 = np.linalg.det(Sigma0)
    det_Sigma1 = np.linalg.det(Sigma1)
    
    # Trace term
    trace_term = np.trace(inv_Sigma1 @ Sigma0)
    
    # Mean difference term
    diff = mu1 - mu0
    mean_term = diff.T @ inv_Sigma1 @ diff
    
    # KL divergence
    kl = 0.5 * (trace_term + mean_term - d + np.log(det_Sigma1 / det_Sigma0))
    return kl

kl_all = []
mean_er_all = []
cov_er_all = []
for i in range(N_iter):
    sample_i = sample_iter[i].numpy()
    mean_i = np.mean(sample_i, axis=0)
    cov_i = np.cov(sample_i.T)
    
    enkf_mean_np = enkf_mean.numpy()
    enkf_cov_np = enkf_cov.numpy()
    
    kl = kl_divergence_gaussians(enkf_mean_np, enkf_cov_np, mean_i, cov_i)
    kl_all.append(kl)
    
    mean_er = np.linalg.norm(mean_i - enkf_mean_np)/np.linalg.norm(enkf_mean_np)
    cov_er = np.linalg.norm(cov_i - enkf_cov_np)/np.linalg.norm(enkf_cov_np)
    mean_er_all.append(mean_er)
    cov_er_all.append(cov_er)
    
    print(f'iter {i}: KL {kl:.4e}, mean er {mean_er:.4e}, cov er {cov_er:.4e}')

kl_ensf = kl_divergence_gaussians(enkf_mean_np, enkf_cov_np, post_sample_score_original.mean(dim=0).numpy(), np.cov(post_sample_score_original.T.numpy()))
mean_er_ensf = np.linalg.norm(post_sample_score_original.mean(dim=0).numpy() - enkf_mean_np)/np.linalg.norm(enkf_mean_np)
cov_er_ensf = np.linalg.norm(np.cov(post_sample_score_original.T.numpy()) - enkf_cov_np)/np.linalg.norm(enkf_cov_np)
print(f'EnSF original: KL {kl_ensf:.4e}, mean er {mean_er_ensf:.4e}, cov er {cov_er_ensf:.4e}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

ax = axes[0]
ax.plot(kl_all, '-o', label='Iterative EnSF')
ax.hlines(kl_ensf, 0, N_iter-1, colors='r', linestyles='dashed', label='EnSF original')
ax.set_ylabel('KL divergence')
ax.set_title('KL divergence VS iteration')


ax = axes[1]
ax.plot(mean_er_all, '-o', label='Iterative EnSF')
ax.hlines(mean_er_ensf, 0, N_iter-1, colors='r', linestyles='dashed', label='EnSF original')
ax.set_ylabel('relative error of mean')
ax.set_title('relative error of mean VS iteration')


ax = axes[2]
ax.plot(cov_er_all, '-o', label='Iterative EnSF')
ax.hlines(cov_er_ensf, 0, N_iter-1, colors='r', linestyles='dashed', label='EnSF original')
ax.set_ylabel('relative error of covariance')
ax.set_title('relative error of covariance VS iteration')

for i in range(3):
    ax = axes[i]
    ax.grid()
    ax.legend()
    ax.set_xlabel('Iteration')
    ax.set_yscale('log')

plt.tight_layout()
# plt.savefig('linear_gaussian_KL_mean_cov.pdf', dpi=200)
# plt.savefig('linear_gaussian_KL_mean_cov.png', dpi=200)
plt.show()

In [ ]:
N_plot = 500
N_iter_plot = 4

fig, axes = plt.subplots(1, 4, figsize=(15, 5))
axes = axes.flatten()

# Scatter of sample
for i in range(N_iter_plot):
    ax = axes[i]
    cs1 = ax.contour(x_2d, y_2d, pdf_prior_2d.reshape(N_mesh, N_mesh), levels=8, colors="blue", linewidths=0.8, alpha=0.9)
    cs2 = ax.contour(x_2d, y_2d, pdf_post_2d.reshape(N_mesh, N_mesh), levels=8, colors="red", linewidths=0.8, alpha=0.9)
    
    cs = ax.scatter(*sample_iter[i][:N_plot].T, zorder=3, marker='.', c='green', linewidths=0, s=30, alpha=0.4)
    #
    h1, _ = cs1.legend_elements()
    h2, _ = cs2.legend_elements()    
    handles = [h1[0], h2[0], cs]
    labels  = ["Prior", "True posterior", "Posterior samples"]
    ax.legend(handles, labels, loc="lower left",fontsize=10)
    #
    ax.set_title(f'IEnSF (iter={i+1})', fontsize=18)
    ax.set_aspect('equal')
    ax.set_xlim(0,4)
    ax.set_ylim(-4,0)
    ax.grid()

plt.tight_layout()
# plt.savefig('linear_gaussian_convergence.pdf', dpi=200)
# plt.savefig('linear_gaussian_convergence.png', dpi=200)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes = axes.flatten()

############## 
ax = axes[0]
ax.plot(np.arange(1,len(kl_all)+1) ,kl_all, '-o', label='IEnSF')
ax.hlines(kl_ensf, 1, N_iter, colors='r', linestyles='dashed', label='EnSF')
ax.set_yscale('log')
ax.set_xlim(0.95, 5.05)
ax.set_xlabel('Iteration',fontsize=12)
ax.set_ylabel('KL divergence',fontsize=12)
ax.set_title('IEnSF (KL VS iter)', fontsize=16)
ax.legend()
ax.grid(which="major", axis="y", linestyle="-", color="gray", alpha=0.7)
ax.grid(which="minor", axis="y", linestyle="--", color="lightgray", alpha=0.7)
ax.grid(which="major", axis="x", linestyle="-", color="gray", alpha=0.7)

##############
ax = axes[1]
cs1 = ax.contour(x_2d, y_2d, pdf_prior_2d.reshape(N_mesh, N_mesh), levels=8, colors="blue", linewidths=0.8, alpha=0.9)
cs2 = ax.contour(x_2d, y_2d, pdf_post_2d.reshape(N_mesh, N_mesh), levels=8, colors="red", linewidths=0.8, alpha=0.9)

cs = ax.scatter(*post_sample_score_original[:N_plot].T, zorder=3, marker='.', c='green', linewidths=0, s=30, alpha=0.4)
#
h1, _ = cs1.legend_elements()
h2, _ = cs2.legend_elements()    
handles = [h1[0], h2[0], cs]
labels  = ["Prior", "True posterior", "Posterior samples"]
ax.legend(handles, labels, loc="lower left",fontsize=8)
#
ax.set_title(f'EnSF', fontsize=16)
ax.set_aspect('equal')
ax.set_xlim(0,4)
ax.set_ylim(-4,0)
ax.grid()

##############
ax = axes[2]
cs1 = ax.contour(x_2d, y_2d, pdf_prior_2d.reshape(N_mesh, N_mesh), levels=8, colors="blue", linewidths=0.8, alpha=0.9)
cs2 = ax.contour(x_2d, y_2d, pdf_post_2d.reshape(N_mesh, N_mesh), levels=8, colors="red", linewidths=0.8, alpha=0.9)
cs = ax.scatter(*sample_iter[4][:N_plot].T, zorder=3, marker='.', c='green', linewidths=0, s=30, alpha=0.4)
#
h1, _ = cs1.legend_elements()
h2, _ = cs2.legend_elements()    
handles = [h1[0], h2[0], cs]
labels  = ["Prior", "True posterior", "Posterior samples"]
ax.legend(handles, labels, loc="lower left",fontsize=8)
#
ax.set_title(f'IEnSF (iter=5)', fontsize=16)
ax.set_aspect('equal')
ax.set_xlim(0,4)
ax.set_ylim(-4,0)
ax.grid()



# plt.savefig('linear_gaussian_KL.pdf', dpi=200)
# plt.savefig('linear_gaussian_KL.png', dpi=200)
plt.show()